# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa – Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset schema entities (record sets, fields, columns, etc.) are referenced by their `@id` for maximum reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

- **FAIR² Mental Health Survey Dataset**: https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant pandas

## 1. Data Loading

Load dataset metadata and access record sets with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata as a dict using .to_json() for inspection
metadata = dataset.metadata.to_json()
print(f"Dataset name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata.get('datePublished', 'N/A')}")

## 2. Data Overview

List and inspect available record sets and their field/column structure via `@id`. Record sets and fields in Croissant are referenced with their `@id` property.

Let's enumerate the record sets and fields available in this dataset.

In [ ]:
record_sets = dataset.metadata.record_sets
print(f"Record sets found: {len(record_sets)}\n")
if not record_sets:
    print("No record sets found in the metadata (possibly a non-tabular or minimally described Croissant file).")
else:
    for rs in record_sets:
        print(f"  RecordSet @id: {rs.id}")
        print(f"         name: {rs.name}")
        print(f"  description: {rs.description}")
        print("     Fields / Columns:")
        if getattr(rs, 'fields', []):
            for field in rs.fields:
                print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type if hasattr(field,'data_type') else 'N/A'}")
        elif getattr(rs, 'columns', []):  # fallback
            for col in rs.columns:
                print(f"    - Column @id: {col.id}, name: {col.name}, dataType: {col.data_type if hasattr(col,'data_type') else 'N/A'}")
        print()

### Example: Preview the first few records from the main record set

Replace `<record_set_id>` with the `@id` of the main tabular record set for interactive preview.


In [ ]:
# Suggest main record set (if present)
if record_sets:
    # Often, the main survey data will be in the first (or the only) record set
    main_record_set = record_sets[0].id
    print(f"Previewing first 2 records from record set {main_record_set}")
    for i, rec in enumerate(dataset.records(record_set=main_record_set)):
        print(json.dumps(rec, indent=2))
        if i >= 1:
            break
else:
    print("No tabular record sets found.")

## 3. Data Extraction

Load the primary table(s) into pandas DataFrames. All references to record sets and fields/columns use their Croissant `@id`.

In [ ]:
# Collect all record set ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
# Load into pandas
for record_set_id in record_set_ids:
    print(f"Loading record set {record_set_id}")
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    print(f"Shape: {df.shape}")
    print(f"Columns (@id): {list(df.columns)}")
    if not df.empty:
        print(df.head(2))
    print()
    dataframes[record_set_id] = df
# Select the main record set for analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main record set: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
else:
    main_record_set_id = None
    main_df = pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate several basic data wrangling and exploration steps. All processing is done by referencing the Croissant field/column `@id`s.

In [ ]:
# Example: Filter, normalize, and group data by field
if not main_df.empty:
    # Suggest probable numeric field (`gad7_score`), adjust if necessary
    probable_numeric_fields = [col for col in main_df.columns if ('gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower() or 'score' in col.lower())]
    numeric_field_id = probable_numeric_fields[0] if probable_numeric_fields else main_df.columns[0]
    print(f"Using numeric field for EDA: {numeric_field_id}\n")

    # Filtering (e.g., only high GAD-7 scores)
    threshold = 10
    if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
    else:
        # Try to coerce to numeric
        filtered_df = main_df.copy()
        filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows\n")
    print(filtered_df.head(3))

    # Normalize column
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, col_norm]].head(3))

    # Suggest a grouping field (e.g., `gender` or the first string field)
    probable_group_fields = [col for col in main_df.columns if 'gender' in col.lower() or 'village' in col.lower() or 'education' in col.lower()]
    group_field_id = probable_group_fields[0] if probable_group_fields else main_df.columns[0]
    if group_field_id in filtered_df.columns:
        print(f"\nGrouped by {group_field_id} (showing normalized mean of {numeric_field_id}):")
        grouped_df = filtered_df.groupby(group_field_id)[[numeric_field_id, col_norm]].mean().sort_values(numeric_field_id, ascending=False)
        print(grouped_df.head(5))
else:
    print('Main DataFrame not loaded (no data).')

## 5. Visualization

Visualize the distribution of selected fields (referenced by their Croissant `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not main_df.empty:
    plt.figure(figsize=(7, 4))
    # Field for histogram
    hist_field = numeric_field_id if 'numeric_field_id' in locals() else main_df.columns[0]
    plt.title(f"Distribution of {hist_field}")
    sns.histplot(main_df[hist_field].dropna(), bins=15, kde=True)
    plt.xlabel(hist_field)
    plt.tight_layout()
    plt.show()

    # Boxplot for grouping field
    plt.figure(figsize=(8, 4))
    if group_field_id in main_df.columns:
        sns.boxplot(x=main_df[group_field_id], y=main_df[hist_field])
        plt.xlabel(group_field_id)
        plt.ylabel(hist_field)
        plt.title(f"{hist_field} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print('No main DataFrame available for visualization.')

## 6. Conclusion

In this notebook, we loaded the FAIR² Mental Health Survey Dataset Croissant schema, programmatically inspected tabular structures by referencing all primary entities via their `@id`, and performed basic exploratory and visual analysis. This approach supports reproducibility and robust data workflows by relying on the Croissant standard and explicit entity references.

You can extend these steps for deeper feature engineering, advanced analyses, machine learning, or data fusion across Croissant-compliant datasets.